## Reddit topic modeling

In [ ]:
import sys
!{sys.executable} -m pip install nltk

In [ ]:
import pandas as pd
import re
from gensim.models import LdaModel, CoherenceModel
from gensim.corpora.dictionary import Dictionary
from tqdm import tqdm
import matplotlib.pyplot as plt
from gensim import corpora, models
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns

# ── Load spaCy (aligned with thesis tokenization pipeline) ──
import spacy
try:
    nlp_en = spacy.load('en_core_web_sm')
    SPACY_OK = True
    print("spaCy (en_core_web_sm) loaded.")
except Exception:
    SPACY_OK = False
    print("spaCy not available — will fall back to regex tokenization.")

# ── Multi-word protected phrases (aligned with thesis SPACY_PROTECTED_PHRASES) ──
SPACY_PROTECTED_PHRASES = [
    # Ghost / supernatural phenomena
    'sleep paralysis', 'shadow person', 'shadow figure',
    'old hag syndrome', 'near death experience',
    'out of body experience', 'out-of-body experience',
    # Hypnagogic / liminal states
    'half awake', 'half asleep', 'half-awake', 'half-asleep',
    # Locations (multi-word)
    'living room', 'school building', 'abandoned building',
    'abandoned house', 'abandoned school',
    # Paranormal investigation tools / methods
    'ouija board', 'spirit box', 'EVP session',
    'ghost hunting', 'ghost hunter', 'paranormal activity',
    'paranormal investigation', 'haunted location',
    'disembodied voice', 'cold touch', 'sage smudging',
    'urban exploration', 'abandoned mansion',
    # Additional multi-word ghost terms
    'hat man', 'white lady',
]

if SPACY_OK:
    for phrase in SPACY_PROTECTED_PHRASES:
        nlp_en.tokenizer.add_special_case(phrase, [{"ORTH": phrase}])
    print(f"  Registered {len(SPACY_PROTECTED_PHRASES)} spaCy protected phrases.")

# ── Affect abbreviations (aligned with thesis AFFECT_ABBREVS) ──
AFFECT_ABBREVS = {
    'WTF':   'what the fuck',
    'OMG':   'oh my god',
    'OMFG':  'oh my fucking god',
    'FFS':   "for fuck's sake",
    'LMAO':  'laughing my ass off',
    'NGL':   'not gonna lie',
    'TBH':   'to be honest',
    'IDK':   "i don't know",
    'IYKYK': 'if you know you know',
    'DOA':   'dead on arrival',
    'RIP':   'rest in peace',
    'IWTD':  'i want to die',
}

# ── Data loading ──
reddit_paranormal_path = "data/raw/reddit_paranormal_articles_all.csv"
reddit_ghoststories_path = "data/raw/reddit_ghoststories_articles_all.csv"

df_paranormal = pd.read_csv(reddit_paranormal_path)
df_ghoststories = pd.read_csv(reddit_ghoststories_path)
merged_df = pd.concat([df_paranormal, df_ghoststories], ignore_index=True)
merged_df.drop_duplicates(subset=['url'], inplace=True)
merged_df.to_csv("merged_reddit_articles.csv", index=False, encoding="utf-8")
print(f"Merged data saved: {merged_df.shape[0]} posts")

merged_df = pd.read_csv("merged_reddit_articles.csv")
text_col = "content" if "content" in merged_df.columns else merged_df.columns[0]

# ── Build stopwords ──
import nltk
from nltk.corpus import stopwords as nltk_stopwords
nltk.download('stopwords', quiet=True)

combined_stopwords = set(nltk_stopwords.words('english'))

try:
    with open('data/raw/stopwords-en.txt', 'r', encoding='utf-8') as f:
        iso_stopwords = set(line.strip() for line in f if line.strip())
except FileNotFoundError:
    iso_stopwords = set()
    print("External stopwords file not found, using base stopwords only.")

combined_stopwords |= iso_stopwords
combined_stopwords = {w for w in combined_stopwords if w.isalpha()}

# Discourse hedges / filler words (aligned with thesis STOPWORDS_EN)
combined_stopwords |= {
    'like','just','really','actually','basically','literally','obviously',
    'maybe','probably','definitely','honestly','apparently','seriously',
}

# ── POS whitelist (aligned with thesis KEEP_POS_EN) ──
KEEP_POS_EN = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}

def normalize_en(text):
    """Expand affect abbreviations before tokenization."""
    t = str(text)
    for abbr, expanded in AFFECT_ABBREVS.items():
        t = re.sub(rf'(?<!\w){re.escape(abbr)}(?!\w)', expanded, t, flags=re.IGNORECASE)
    return t

def preprocess_text(text):
    text = normalize_en(text)
    if SPACY_OK:
        doc = nlp_en(text.lower())
        tokens = [
            token.lemma_ for token in doc
            if token.pos_ in KEEP_POS_EN
            and token.lemma_ not in combined_stopwords
            and len(token.lemma_) > 2
            and token.is_alpha
        ]
        return tokens
    else:
        # Regex fallback
        tokens = re.findall(r'\b[a-z]{2,}\b', text.lower())
        return [w for w in tokens if w not in combined_stopwords]

merged_df['cleaned_text'] = merged_df[text_col].apply(lambda x: ' '.join(preprocess_text(x)))
print("Text preprocessing complete (spaCy pipeline).")


In [ ]:
from gensim import corpora, models
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Tokenize cleaned text for topic modeling
tokenized_texts = merged_df['cleaned_text'].apply(word_tokenize).tolist()

# ==================== 2. Dictionary and Corpus ====================
dictionary = corpora.Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

# ==================== 3. Train with multiple topic numbers ====================
coherence_scores = []
perplexity_scores = []
topic_nums = list(range(2, 11))  # Try 2 to 10 topics

print(" Training LDA models with different topic numbers...")
for k in tqdm(topic_nums):
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )
    
    # Coherence
    coherence_model = CoherenceModel(model=lda, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
    coherence = coherence_model.get_coherence()
    coherence_scores.append(coherence)

    # Perplexity
    perplexity = lda.log_perplexity(corpus)
    perplexity_scores.append(perplexity)

print(" All models trained.")

# ==================== 4. Plot results ====================
plt.figure(figsize=(12, 5))

# Coherence plot
plt.subplot(1, 2, 1)
plt.plot(topic_nums, coherence_scores, marker='o')
plt.title('Coherence Score vs Number of Topics')
plt.xlabel('Num Topics')
plt.ylabel('Coherence Score')

# Perplexity plot
plt.subplot(1, 2, 2)
plt.plot(topic_nums, perplexity_scores, marker='o', color='orange')
plt.title('Perplexity Score vs Number of Topics')
plt.xlabel('Num Topics')
plt.ylabel('Perplexity (log)')

plt.tight_layout()
plt.show()

# ==================== 5. Save results to CSV ====================
score_df = pd.DataFrame({
    'num_topics': topic_nums,
    'coherence': coherence_scores,
    'perplexity': perplexity_scores
})
score_df.to_csv("lda_scores_by_topic_count.csv", index=False, encoding="utf-8")
print("Saved score results to lda_scores_by_topic_count.csv")


choose num=7 topics

In [ ]:
from gensim.models import LdaModel
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
import pandas as pd

# 確保你已經有以下這些變數：
# tokenized_texts, corpus, dictionary, merged_df

# ==================== 1. Train LDA with 7 topics ====================
num_topics = 7

lda_model_7 = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# ==================== 2. Show Top Words per Topic ====================
print("\n Top words per topic:")
topics = lda_model_7.print_topics(num_words=10)
for topic_id, topic_words in topics:
    print(f"Topic {topic_id}: {topic_words}")

# ==================== 3. Add topic distribution to each document ====================
def get_topic_distribution(bow):
    topic_dist = lda_model_7.get_document_topics(bow, minimum_probability=0)
    return [prob for _, prob in topic_dist]

topic_distributions = [get_topic_distribution(doc) for doc in corpus]
topic_df = pd.DataFrame(topic_distributions, columns=[f"Topic_{i}" for i in range(num_topics)])
merged_with_topics = pd.concat([merged_df.reset_index(drop=True), topic_df], axis=1)

# ==================== 4. Save to CSV ====================
output_file = "reddit_ghoststories_topics_7.csv"
merged_with_topics.to_csv(output_file, index=False, encoding="utf-8")
print(f"\n Saved topic distribution to {output_file}")


In [ ]:
# For each document, get the dominant topic (highest probability)
merged_with_topics['dominant_topic'] = topic_df.idxmax(axis=1)
print(" Added 'dominant_topic' column to DataFrame.")


In [ ]:
# Extract top keywords for each topic to use as labels
topic_keywords = {}
for topic_id, topic_words in lda_model_7.show_topics(num_topics=num_topics, num_words=5, formatted=False):
    label = ", ".join([word for word, _ in topic_words])
    topic_keywords[f"Topic_{topic_id}"] = label

# Map topic number to keyword labels
merged_with_topics['topic_label'] = merged_with_topics['dominant_topic'].map(topic_keywords)
print("Added 'topic_label' column to DataFrame.")


In [ ]:
merged_with_topics.to_csv("reddit_ghoststories_topics_7_labeled.csv", index=False, encoding="utf-8")
print("Saved enriched results with dominant topics and labels to reddit_ghoststories_topics_7_labeled.csv")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 統計每個主題的出現次數
topic_counts = merged_with_topics['dominant_topic'].value_counts().sort_index()
topic_labels = [topic_keywords[t] for t in topic_counts.index]

# 繪圖
plt.figure(figsize=(12, 6))
sns.barplot(x=topic_labels, y=topic_counts.values, palette="viridis")
plt.xticks(rotation=45, ha='right')
plt.xlabel("Topic (Top Keywords)")
plt.ylabel("Number of Stories")
plt.title(" Dominant Topic Distribution in Reddit Ghost Stories")
plt.tight_layout()
plt.show()


In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()
vis_data = gensimvis.prepare(lda_model_7, corpus, dictionary)
pyLDAvis.save_html(vis_data, "lda_7_topics_viz.html")
print("pyLDAvis saved to lda_7_topics_viz.html (open in browser)")


## Theqoo topic modeling

In [ ]:
from kiwipiepy import Kiwi
import pandas as pd
from collections import Counter, defaultdict
import plotly.express as px
df_ko = pd.read_csv("data/raw/theqoo_experience_articles_v2.csv")  
print(df_ko.head())
text_col = "content" 

In [ ]:
from kiwipiepy import Kiwi
import requests

# ── Korean 초성 abbreviation map (aligned with thesis KOREAN_CHOSUNG_MAP) ──
KOREAN_CHOSUNG_MAP = {
    'ㄷㄷ':   '덜덜',       # trembling / shivering in fear
    'ㅎㄷㄷ': '후덜덜',     # greatly trembling (more intense)
    'ㅂㄷ':   '벌벌',       # shaking
    'ㄷㄱ':   '덜컹',       # sudden thud sound
    'ㅆㅂ':   '씨발',       # strong expletive — peak distress marker
    'ㅁㅊ':   '미쳤어',     # "this is insane"
    'ㄱㅎ':   '공황',       # panic
    'ㅈㅅ':   '죄송',       # sorry
    'ㄱㅅ':   '감사',       # thanks
    'ㅇㅇ':   '응응',       # yes / affirmative
    'ㄴㄴ':   '노노',       # no no
}

def normalize_ko(text):
    """Expand 초성 abbreviations before tokenization."""
    t = str(text)
    # Longest-first to avoid partial matches
    for raw, expanded in sorted(KOREAN_CHOSUNG_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        t = t.replace(raw, expanded)
    return t

# 1. Download spikeekips stopword list
stopword_url = "https://gist.githubusercontent.com/spikeekips/40eea22ef4a89f629abd87eed535ac6a/raw/4f7a635040442a995568270ac8156448f2d1f0cb/stopwords-ko.txt"
stopwords_ko = set()
try:
    r = requests.get(stopword_url)
    r.raise_for_status()
    stopwords_ko = set(line.strip() for line in r.text.splitlines() if line.strip())
    print(f"Loaded {len(stopwords_ko)} Korean stopwords from spikeekips gist.")
except Exception as e:
    print("Failed to download stopwords. Check URL or your internet connection.")
    stopwords_ko = set()

# 2. Load your own stopword list
with open('stopwords-ko.txt', encoding='utf-8') as f:
    stopwords_ko_2 = set(line.strip() for line in f if line.strip())
print(f"Loaded {len(stopwords_ko_2)} Korean stopwords from your file.")

# 3. Load second stopword list (aligned with thesis STOPWORDS_KO)
try:
    with open('stopwords-ko_v2.txt', encoding='utf-8') as f:
        stopwords_ko_v2 = set(line.strip() for line in f if line.strip())
    print(f"Loaded {len(stopwords_ko_v2)} Korean stopwords from stopwords-ko_v2.txt.")
except FileNotFoundError:
    stopwords_ko_v2 = set()
    print("stopwords-ko_v2.txt not found, skipping.")

# 4. Custom chat/filler/discourse words (aligned with thesis STOPWORDS_KO)
stopwords_custom = {
    # Chosung abbreviations
    "ㅎㅎ","ㅋㅋ","ㅠㅠ","ㅜㅜ",
    # Morpheme / particle noise
    "아","음","응","야","헉","어","뭐","것","좀","네","아니",
    "그래","그럼","왜","어떻게","어디","언제","님","수","어요","나오","서",
    # Discourse connectors (from thesis)
    "그래서","그런데","그러면","하지만","그리고","또한","즉","왜냐하면","왜냐면",
    "어떤","이런","저런","아무","정말","진짜","매우","너무","아주","누구",
}

# 5. Combine all stopwords
stopwords_all = stopwords_ko | stopwords_ko_2 | stopwords_ko_v2 | stopwords_custom
print(f"Total unique stopwords: {len(stopwords_all)}")

# 6. Register Kiwi custom compound nouns (aligned with thesis KIWI_CUSTOM_WORDS)
kiwi = Kiwi()
KIWI_CUSTOM_WORDS = [
    # Ghost compound nouns (aligned with thesis)
    '처녀귀신','총각귀신','달걀귀신','손각시','지박령','부유령','동자귀신',
    '귀신들림','빨간마스크','화장실귀신','남자귀신','여자귀신','남귀신','여귀신',
    '귀신이야기','귀신소리','귀신공포','귀신출몰','몽달귀신','물귀신','머슴귀신',
    # Sound / sensation terms (from W2V LLR analysis)
    '가위눌림','울음소리','발소리','속삭임','휘파람',
]
for w in KIWI_CUSTOM_WORDS:
    kiwi.add_user_word(w, 'NNG')
kiwi.add_user_word('소리치', 'VV')
print(f"Registered {len(KIWI_CUSTOM_WORDS)} Kiwi custom compound nouns.")

# 7. Morpheme extraction with ALL stopwords
def get_kiwi_morphs_no_stopwords(text):
    if pd.isna(text): return []
    try:
        text = normalize_ko(str(text))  # expand 초성 abbreviations first
        return [token.form for token in kiwi.tokenize(text) if token.form not in stopwords_all]
    except Exception:
        return []

print("Tokenizing texts using Kiwi (with stopword removal)...")
df_ko["morphemes"] = df_ko[text_col].apply(get_kiwi_morphs_no_stopwords)
print("Morpheme extraction (without stopwords) complete!")


In [ ]:
from gensim import corpora, models
from gensim.models import CoherenceModel
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from nltk.tokenize import word_tokenize

# 假設你已經有斷詞後的 tokenized_texts（df_ko["morphemes"]）：
tokenized_texts = df_ko["morphemes"].tolist()

# ==================== 1. Dictionary and Corpus ====================
dictionary = corpora.Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

# ==================== 2. Train with multiple topic numbers ====================
coherence_scores = []
perplexity_scores = []
topic_nums = list(range(2, 11))  # Try 2 to 10 topics

print(" Training LDA models with different topic numbers...")
for k in tqdm(topic_nums):
    lda = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )
    
    # Coherence
    coherence_model = CoherenceModel(model=lda, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
    coherence = coherence_model.get_coherence()
    coherence_scores.append(coherence)

    # Perplexity
    perplexity = lda.log_perplexity(corpus)
    perplexity_scores.append(perplexity)

print(" All models trained.")

# ==================== 3. Plot results ====================
plt.figure(figsize=(12, 5))

# Coherence plot
plt.subplot(1, 2, 1)
plt.plot(topic_nums, coherence_scores, marker='o')
plt.title('Coherence Score vs Number of Topics (TheQoo)')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score')

# Perplexity plot
plt.subplot(1, 2, 2)
plt.plot(topic_nums, perplexity_scores, marker='o', color='orange')
plt.title('Perplexity Score vs Number of Topics (TheQoo)')
plt.xlabel('Number of Topics')
plt.ylabel('Perplexity (log)')

plt.tight_layout()
plt.show()

# ==================== 4. Save results to CSV ====================
score_df = pd.DataFrame({
    'num_topics': topic_nums,
    'coherence': coherence_scores,
    'perplexity': perplexity_scores
})
score_df.to_csv("theqoo_lda_scores_by_topic_count.csv", index=False, encoding="utf-8-sig")
print(" Saved score results to theqoo_lda_scores_by_topic_count.csv")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
from gensim.models import LdaModel
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 設定韓文字體：AppleGothic（macOS）或 NanumGothic（Linux）
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Linux':
    plt.rcParams['font.family'] = 'NanumGothic'
else:  # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'

# 解決 minus 號顯示錯誤問題
plt.rcParams['axes.unicode_minus'] = False

# ==================== 6. Train Final LDA Model (num_topics = 8) ====================
num_topics = 8
lda_model_ko = models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# ==================== 7. Add Topic Distributions ====================
def get_topic_distribution(bow):
    topic_dist = lda_model_ko.get_document_topics(bow, minimum_probability=0)
    return [prob for _, prob in topic_dist]

topic_distributions = [get_topic_distribution(doc) for doc in corpus]
topic_df = pd.DataFrame(topic_distributions, columns=[f"Topic_{i}" for i in range(num_topics)])
df_ko_with_topics = pd.concat([df_ko.reset_index(drop=True), topic_df], axis=1)

# ==================== 8. Dominant Topic & Label ====================
df_ko_with_topics['dominant_topic'] = topic_df.idxmax(axis=1)

topic_keywords = {}
for topic_id, topic_words in lda_model_ko.show_topics(num_topics=num_topics, num_words=5, formatted=False):
    label = ", ".join([word for word, _ in topic_words])
    topic_keywords[f"Topic_{topic_id}"] = label

df_ko_with_topics['topic_label'] = df_ko_with_topics['dominant_topic'].map(topic_keywords)

# Save labeled result
output_path = f"theqoo_ghoststories_topics_{num_topics}_labeled.csv"
df_ko_with_topics.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved labeled topic distribution to {output_path}")

# ==================== 9. Plot Topic Frequency ====================
topic_counts = df_ko_with_topics['dominant_topic'].value_counts().sort_index()
topic_labels = [topic_keywords[t] for t in topic_counts.index]

plt.figure(figsize=(12, 6))
sns.barplot(x=topic_labels, y=topic_counts.values, palette="viridis")
plt.xticks(rotation=45, ha='right')
plt.xlabel("Topic (Top Keywords)")
plt.ylabel("Number of Stories")
plt.title("Dominant Topic Distribution in TheQoo Ghost Stories")
plt.tight_layout(); plt.show()

# ==================== 10. pyLDAvis ====================
pyLDAvis.enable_notebook()
vis_data = gensimvis.prepare(lda_model_ko, corpus, dictionary)
vis_output_path = f"theqoo_lda_{num_topics}_topics_viz.html"
pyLDAvis.save_html(vis_data, vis_output_path)
print(f"pyLDAvis saved to {vis_output_path}")


## PTT Topic modeling

In [ ]:
# ==================== 0. 安裝必要套件 ====================
import sys
!{sys.executable} -m pip install --upgrade jieba opencc-python-reimplemented

# ==================== 1. 載入 & 整理停用詞 ====================
import jieba
import nltk
from opencc import OpenCC
from TCSP import read_stopwords_list

nltk.download('stopwords')
from nltk.corpus import stopwords

# Combine stopwords: TCSP + NLTK（簡體轉繁體）+ 自訂
my_stopwords_trad = set(read_stopwords_list())
nltk_stopwords_simp = set(stopwords.words('chinese'))
cc = OpenCC('s2t')
nltk_stopwords_trad = set(cc.convert(word) for word in nltk_stopwords_simp)

customized_stopwords_trad = set([
    # PTT boilerplate noise
    "未經授", "權者", "文章", "商業", "用途", "分享", "轉載",
    "恕", "原文", "手機", "排版", "請見諒", "經驗文", "真的", "住",
    "發文", "文章代碼", "時間", "看板", "標題", "作者",
    # Interjections
    "嗨", "哈", "哈哈", "呵呵", "嘻嘻", "嘿嘿", "哎呀", "啊呀",
    "哦", "嗯", "啊", "喔", "唉", "哼", "哇", "呀", "咦", "哎哟",
])

# Baseline fallback (aligned with thesis — guaranteed even if TCSP fails)
baseline_zh = {
    '的','了','在','是','們',
    '和','與','或','但','而','也','都','很','就','到','從',
    '這','那','有','一','說','看','去','來',
    '什麼','怎麼','哪','誰','為什麼','因為','所以','雖然','如果',
}

all_stopwords_trad = my_stopwords_trad | nltk_stopwords_trad | customized_stopwords_trad | baseline_zh
print(f" Combined stopwords: {len(all_stopwords_trad)}")

# ==================== 2. 載入資料 & jieba 斷詞 ====================
import pandas as pd

# Register jieba custom words (aligned with thesis JIEBA_CUSTOM_WORDS)
JIEBA_CUSTOM_WORDS = [
    # Ghost types
    '女鬼','男鬼','紅衣女鬼','魔神仔','好兄弟','水鬼','嬰靈',
    '鬼壓床','鬼打牆','陰陽眼','撞鬼','見鬼','鬼影幢幢',
    # Compound phenomena
    '靈異事件','超自然現象','靈魂出竅','農曆七月',
    '中元節','中元普渡','冤親債主','附身事件',
    # Practitioners
    '陰陽師','風水師傅','通靈師','驅魔師','乩童',
    # Locations
    '廢棄醫院','廢棄學校','廢棄旅館','民雄鬼屋','鬼屋',
    # Bodily sensations
    '雞皮疙瘩','毛骨悚然','鬼壓身',
    # Ritual objects
    '符咒','八卦鏡','桃木劍',
    # Taiwanese folk religion / ritual
    '觀落陰','地基主','拜地基主','孤魂野鬼','招魂術',
    '陰陽界','前世冤孽','靈媒師','通靈儀式','民間信仰',
    '清明節','普渡祭祀','祭祖祭拜','超渡儀式','法會','法事',
    # Sound / appearance / sensation keywords (from W2V LLR analysis)
    '靈異','紅衣小女孩','山靈',
    '哭聲','口哨','腳步聲','低語','呢喃','耳語','笑聲','尖叫',
    '哭泣','呻吟','嘆息','呼吸聲','敲門聲','拍打聲','嚎啕大哭',
    '長頭髮','短頭髮','紅衣','白衣','黑影','白影','洋裝',
    '旗袍','收驚','軍徽','軍卡','守衛室','保健室',
]
for w in JIEBA_CUSTOM_WORDS:
    jieba.add_word(w)
print(f"Registered {len(JIEBA_CUSTOM_WORDS)} jieba custom words.")

# Load external ghost-story custom words TXT if available
import os
_custom_txt = os.path.expanduser('~/Downloads/ghost-story-gender-age-analysis/custom_words_ghost_analysis_traditional_chinese.txt')
if os.path.exists(_custom_txt):
    with open(_custom_txt, encoding='utf-8') as f:
        _ext_words = [ln.strip() for ln in f if ln.strip() and not ln.startswith('#')]
    for w in _ext_words:
        jieba.add_word(w)
    print(f"Loaded {len(_ext_words)} additional jieba words from custom TXT.")

df_zh = pd.read_csv('data/raw/new_ptt_experience_articles_all.csv')
text_col = "content"

def preprocess_and_remove_stopwords(text, stopwords_set):
    if not isinstance(text, str):
        return []
    tokens = jieba.lcut(text)
    return [t for t in tokens if t not in stopwords_set and t.strip() and any('\u4e00' <= c <= '\u9fff' for c in t)]

df_zh["tokens"] = df_zh[text_col].apply(lambda x: preprocess_and_remove_stopwords(x, all_stopwords_trad))
print(" Tokenization and stopword removal complete!")


In [ ]:
# ==================== 3. 建立字典與語料庫 ====================
from gensim import corpora, models
from gensim.models import CoherenceModel
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

tokenized_texts = df_zh["tokens"].tolist()
dictionary = corpora.Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

# ==================== 4. 主題數 sweep：coherence & perplexity ====================
coherence_scores = []
perplexity_scores = []
topic_nums = list(range(2, 11))

for k in tqdm(topic_nums):
    lda = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )
    coherence_model = CoherenceModel(model=lda, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
    coherence_scores.append(coherence_model.get_coherence())
    perplexity_scores.append(lda.log_perplexity(corpus))

score_df = pd.DataFrame({
    'num_topics': topic_nums,
    'coherence': coherence_scores,
    'perplexity': perplexity_scores
})
score_df.to_csv("ptt_lda_scores_by_topic_count.csv", index=False, encoding="utf-8-sig")
print(" Saved score results to ptt_lda_scores_by_topic_count.csv")


# ==================== 6. 畫出分數圖 ====================
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(topic_nums, coherence_scores, marker='o')
plt.title('Coherence Score vs Number of Topics (PTT)')
plt.xlabel('number of topics'); plt.ylabel('Coherence Score')

plt.subplot(1, 2, 2)
plt.plot(topic_nums, perplexity_scores, marker='o', color='orange')
plt.title('Perplexity vs Number of Topics (PTT)')
plt.xlabel('number of topics'); plt.ylabel('Perplexity (log)')
plt.tight_layout(); plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 你的字型檔案位置
font_path = "Noto_Sans_TC (1)/NotoSansTC-VariableFont_wght.ttf"

# 載入字型
font_prop = fm.FontProperties(fname=font_path)

# 設定為 matplotlib 預設字型
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform


num_topics = 10
lda_model_zh = models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# ==================== 8. 加入主題分佈 ====================
def get_topic_distribution(bow):
    topic_dist = lda_model_zh.get_document_topics(bow, minimum_probability=0)
    return [prob for _, prob in topic_dist]

topic_distributions = [get_topic_distribution(doc) for doc in corpus]
topic_df = pd.DataFrame(topic_distributions, columns=[f"Topic_{i}" for i in range(num_topics)])
df_zh_with_topics = pd.concat([df_zh.reset_index(drop=True), topic_df], axis=1)

# ==================== 9. 主題標籤與 dominant topic ====================
df_zh_with_topics['dominant_topic'] = topic_df.idxmax(axis=1)

topic_keywords = {}
for topic_id, topic_words in lda_model_zh.show_topics(num_topics=num_topics, num_words=5, formatted=False):
    label = ", ".join([word for word, _ in topic_words])
    topic_keywords[f"Topic_{topic_id}"] = label

df_zh_with_topics['topic_label'] = df_zh_with_topics['dominant_topic'].map(topic_keywords)

# 儲存結果
output_path = f"new_ptt_ghoststories_topics_{num_topics}_labeled.csv"
df_zh_with_topics.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f" Saved labeled result to {output_path}")

# ==================== 10. 主題分佈長條圖 ====================
topic_counts = df_zh_with_topics['dominant_topic'].value_counts().sort_index()
topic_labels = [topic_keywords[t] for t in topic_counts.index]

plt.figure(figsize=(12, 6))
sns.barplot(x=topic_labels, y=topic_counts.values, palette="viridis")
plt.xticks(rotation=45, ha='right')
plt.xlabel("Top keywords for topic")
plt.ylabel("number of stories")
plt.title(" PTT Ghost Stories Dominant Topic Distribution")
plt.tight_layout(); plt.show()

# ==================== 11. pyLDAvis 主題互動視覺化 ====================
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()
vis_data = gensimvis.prepare(lda_model_zh, corpus, dictionary)
vis_output_path = f"new_ptt_lda_{num_topics}_topics_viz.html"
pyLDAvis.save_html(vis_data, vis_output_path)
print(f" pyLDAvis saved to {vis_output_path}")


K-Means for topic mapping

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

# Load the datasets with topic labels
df_reddit = pd.read_csv("reddit_ghoststories_topics_7_labeled.csv")  # Reddit dataset (already loaded)
df_theqoo = pd.read_csv("theqoo_ghoststories_topics_8_labeled.csv")
df_ptt = pd.read_csv("new_ptt_ghoststories_topics_10_labeled.csv")

# Extract topic info from each dataset
def extract_topic_keywords(df, lang):
    topic_keywords = df[['dominant_topic', 'topic_label']].drop_duplicates()
    topic_keywords['lang'] = lang
    topic_keywords['topic_id'] = topic_keywords['dominant_topic'].astype(str)
    topic_keywords['label_text'] = topic_keywords['topic_label'].astype(str)
    return topic_keywords[['topic_id', 'label_text', 'lang']]

# Extract topic keywords from each dataset
reddit_topics = extract_topic_keywords(df_reddit, 'en')
theqoo_topics = extract_topic_keywords(df_theqoo, 'ko')
ptt_topics = extract_topic_keywords(df_ptt, 'zh')

# Combine all topics
topic_all_langs = pd.concat([reddit_topics, theqoo_topics, ptt_topics], ignore_index=True)
print(f"Total topics across all languages: {len(topic_all_langs)}")

# Load multilingual embedding model
model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# Encode topic labels as vectors
topic_all_langs["embedding"] = topic_all_langs["label_text"].apply(lambda x: model.encode(x))

# Convert embeddings to numpy array for clustering
X = np.vstack(topic_all_langs["embedding"].values)

# Apply K-means clustering
k = 5  # Number of cross-language topic clusters
kmeans = KMeans(n_clusters=k, random_state=42)
topic_all_langs["cluster"] = kmeans.fit_predict(X)

# Add forum source for better visualization
topic_all_langs["forum"] = topic_all_langs["lang"].map({"en": "Reddit", "ko": "TheQoo", "zh": "PTT"})

# Visualize the distribution of topics across clusters
plt.figure(figsize=(12, 6))
sns.countplot(data=topic_all_langs, x="cluster", hue="forum", palette="Set2")
plt.title("Cross-Language Topic Cluster Distribution")
plt.xlabel("Cluster (Semantically Similar Topics)")
plt.ylabel("Number of Topics")
plt.legend(title="Forum")
plt.tight_layout()
plt.show()

# Display the topics in each cluster
for cluster_id in range(k):
    print(f"\nCluster {cluster_id}:")
    cluster_topics = topic_all_langs[topic_all_langs["cluster"] == cluster_id]
    for _, row in cluster_topics.iterrows():
        print(f"  {row['forum']} ({row['topic_id']}): {row['label_text']}")

# Save the results
topic_all_langs.to_csv("cross_forum_topic_mapping.csv", index=False, encoding="utf-8-sig")
print("\nSaved cross-forum topic mapping to cross_forum_topic_mapping.csv")


In [ ]:
topic_all_langs.to_csv("crosslingual_topic_mapping_kmeans.csv", index=False, encoding='utf-8-sig')


---
## Topic-to-Theme Mapping: Justification for LLR Thematic Grouping

The following section maps LDA-derived topic clusters to the three thematic categories  
used in the LLR collocate analysis: **Space**, **Colour/Appearance**, and **Haunting Methods**.  

This provides a **data-driven justification** for the thematic grouping applied to LLR-significant collocates,  
demonstrating that these categories are not arbitrary but emerge naturally from the corpus via unsupervised topic modeling.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOPIC-TO-THEME MAPPING
# Step 1: Display all K-Means clusters with their LDA topic keywords
# Step 2: Manually annotate which clusters correspond to Space / Colour / Haunting
# Step 3: Generate a summary table for thesis
# ─────────────────────────────────────────────────────────────────────────────

# --- Step 1: Show all clusters with full keyword context ---
print('=' * 70)
print('  K-MEANS CLUSTERS → LDA TOPIC KEYWORDS')
print('=' * 70)
for cluster_id in sorted(topic_all_langs['cluster'].unique()):
    cluster_topics = topic_all_langs[topic_all_langs['cluster'] == cluster_id]
    print(f'\nCluster {cluster_id}:')
    for _, row in cluster_topics.iterrows():
        print(f"  [{row['forum']:7s}] {row['topic_id']:10s} → {row['label_text']}")

# --- Step 2: Define cluster-to-theme mapping ---
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  INSTRUCTIONS:                                                       ║
# ║  After reviewing the cluster output above, fill in the mapping       ║
# ║  below. Map each K-Means cluster to one of your thematic categories  ║
# ║  based on the dominant topic keywords in that cluster.               ║
# ║  Clusters that don't fit any theme can be mapped to 'other'.         ║
# ╚═══════════════════════════════════════════════════════════════════════╝

CLUSTER_TO_THEME = {
    # --- UPDATE THESE based on your actual cluster output ---
    0: 'indoor & domestic hauntings',
    1: 'outdoor',
    2: 'sleep paralysis & dream-related',
    3: 'interpersonal relationships',
    4: 'family relationships',
}

if CLUSTER_TO_THEME:
    topic_all_langs['theme_category'] = topic_all_langs['cluster'].map(CLUSTER_TO_THEME)

    # --- Step 3: Summary table for thesis ---
    print('\n' + '=' * 70)
    print('  THEME CATEGORY MAPPING SUMMARY')
    print('=' * 70)
    for theme in ['indoor & domestic hauntings', 'outdoor', 'sleep paralysis & dream-related', 'interpersonal relationships', 'family relationships']:
        subset = topic_all_langs[topic_all_langs['theme_category'] == theme]
        if subset.empty:
            continue
        print(f'\n  [{theme.upper()}]')
        for _, row in subset.iterrows():
            print(f"    {row['forum']:7s} {row['topic_id']:10s} → {row['label_text']}")

    # Count how many platforms contribute to each theme
    print('\n' + '=' * 70)
    print('  CROSS-PLATFORM COVERAGE PER THEME')
    print('=' * 70)
    coverage = topic_all_langs.groupby('theme_category')['forum'].apply(
        lambda x: ', '.join(sorted(x.unique()))
    )
    for theme, forums in coverage.items():
        print(f'  {theme:12s} → {forums}')

    # Save
    out_path = 'topic_to_theme_mapping.csv'
    topic_all_langs.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'\nSaved: {out_path}')
else:
    print('\n[ACTION REQUIRED] Fill in CLUSTER_TO_THEME mapping above after reviewing clusters.')


---
### Methodological Note for Thesis

The following paragraph can be adapted for inclusion in Chapter 3 (Methodology):

> To establish a data-driven basis for the thematic grouping of LLR-significant collocates, we first conducted unsupervised Latent Dirichlet Allocation (LDA) topic modeling independently on each platform corpus (Reddit: *k* topics; Theqoo: *k* topics; PTT: *k* topics), with the number of topics selected based on coherence score maximisation. Topic keywords from all three platforms were then embedded using a multilingual sentence transformer (distiluse-base-multilingual-cased-v1) and clustered via K-Means (*k* = 7) to identify cross-lingual thematic correspondences. The resulting clusters converged around three dominant narrative dimensions — spatial settings, visual appearance/colour, and haunting methods — which were subsequently adopted as the thematic categories for LLR collocate grouping. This two-stage approach (LDA → cross-lingual clustering) ensures that the thematic categories reflect patterns emergent from the data rather than imposed *a priori* by the researcher.

**Remember to fill in the actual topic numbers (*k*) for each platform.**